# DDRI Bike-Change 군집별 baseline 비교

## 용어 설명

- `cluster baseline`(군집별 기준선): 각 군집을 따로 떼어 동일한 모델로 먼저 성능을 보는 비교
- `cluster00~04`: 기존 대표 15개 군집 라벨
- `sign_accuracy`(부호 일치율): 예측값과 실제값의 증가/감소 방향이 같은 비율

목표:
- `bike_change`에서 군집별 난이도 차이가 얼마나 나는지 본다.
- 이후 군집별 개별 실험 우선순위를 정하는 기준선을 만든다.


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'

TRAIN_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_test_2025.csv'
METRICS_PATH = OUTPUT_DATA_DIR / 'ddri_bike_change_cluster_baseline_metrics.csv'


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

FEATURE_COLUMNS = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h', 'is_weekend', 'is_night_hour',
    'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos', 'commute_morning_flag',
    'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
    'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]
CATEGORICAL_COLUMNS = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_2023 = train_df[train_df['date'] < '2024-01-01'].copy()
valid_2024 = train_df[train_df['date'] >= '2024-01-01'].copy()
full_train = train_df.copy()


In [3]:
def evaluate_predictions(cluster_id: int, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'cluster': cluster_id,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_xy(df: pd.DataFrame):
    X = df[FEATURE_COLUMNS].copy()
    for col in CATEGORICAL_COLUMNS:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def build_lgbm_model():
    return LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='regression',
    )


In [4]:
results = []
cluster_station_map = (
    train_df[['cluster', 'station_group', 'station_id']]
    .drop_duplicates()
    .groupby(['cluster', 'station_group'])['station_id']
    .nunique()
    .reset_index(name='station_count')
)

for cluster_id in sorted(train_df['cluster'].unique()):
    train_cluster = train_2023[train_2023['cluster'] == cluster_id].copy()
    valid_cluster = valid_2024[valid_2024['cluster'] == cluster_id].copy()
    full_cluster = full_train[full_train['cluster'] == cluster_id].copy()
    test_cluster = test_df[test_df['cluster'] == cluster_id].copy()

    if min(len(train_cluster), len(valid_cluster), len(test_cluster)) == 0:
        continue

    X_train, y_train = prepare_xy(train_cluster)
    X_valid, y_valid = prepare_xy(valid_cluster)
    X_full, y_full = prepare_xy(full_cluster)
    X_test, y_test = prepare_xy(test_cluster)

    model = build_lgbm_model()
    model.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
    results.append(evaluate_predictions(cluster_id, 'train_2023', y_train, model.predict(X_train)))
    results.append(evaluate_predictions(cluster_id, 'validation_2024', y_valid, model.predict(X_valid)))

    model = build_lgbm_model()
    model.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
    results.append(evaluate_predictions(cluster_id, 'train_full_refit', y_full, model.predict(X_full)))
    results.append(evaluate_predictions(cluster_id, 'test_2025_refit', y_test, model.predict(X_test)))

results_df = pd.DataFrame(results).merge(cluster_station_map, on='cluster', how='left')
results_df.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
print('saved:', METRICS_PATH)
results_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002926 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34
[LightGBM] [Info] Start training from score 0.000039


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005193 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1747
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34
[LightGBM] [Info] Start training from score -0.000019


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1860
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34
[LightGBM] [Info] Start training from score 0.000078


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003044 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1905
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34
[LightGBM] [Info] Start training from score 0.000019


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009043 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1650
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009420 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1688
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34
[LightGBM] [Info] Start training from score -0.000019


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.095503 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1680
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020965 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1700
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_cluster_baseline_metrics.csv


,cluster,split,rmse,mae,r2,sign_accuracy,station_group,station_count
0,0,train_2023,0.7720,0.5336,0.6608,0.9065,업무/상업 혼합형,3
1,0,validation_2024,0.8900,0.6006,0.4594,0.8861,업무/상업 혼합형,3
2,0,train_full_refit,0.8087,0.5549,0.5937,0.8973,업무/상업 혼합형,3
3,0,test_2025_refit,1.0772,0.6106,0.2911,0.9118,업무/상업 혼합형,3
4,1,train_2023,1.1409,0.7575,0.7643,0.8708,아침 도착 업무 집중형,3
5,1,validation_2024,1.5540,0.9177,0.5789,0.8541,아침 도착 업무 집중형,3
6,1,train_full_refit,1.2509,0.8073,0.7221,0.8745,아침 도착 업무 집중형,3
7,1,test_2025_refit,1.5691,0.9525,0.6255,0.8658,아침 도착 업무 집중형,3
8,2,train_2023,0.6230,0.4117,0.6792,0.9292,주거 도착형,3
9,2,validation_2024,0.8377,0.5147,0.4882,0.9065,주거 도착형,3


In [5]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,cluster,split,rmse,mae,r2,sign_accuracy,station_group,station_count
0,3,test_2025_refit,0.6522,0.4515,0.4704,0.9178,생활권 혼합형,3
1,4,test_2025_refit,0.8631,0.5979,0.5509,0.8883,외곽 주거형,3
2,2,test_2025_refit,0.8634,0.5729,0.5313,0.9027,주거 도착형,3
3,0,test_2025_refit,1.0772,0.6106,0.2911,0.9118,업무/상업 혼합형,3
4,1,test_2025_refit,1.5691,0.9525,0.6255,0.8658,아침 도착 업무 집중형,3
